# Sinh prediction ViT5-Base trên `test_core_2000.jsonl` bằng 2 GPU T4

Notebook này dùng mô hình `VietAI/vit5-base-vietnews-summarization` và cấu hình `beam4_lp1.1_max128_nr3`:

- `num_beams=4`
- `length_penalty=1.1`
- `max_new_tokens=128`
- `no_repeat_ngram_size=3`
- `do_sample=False`
- `seed=42`
- **Không thêm prefix** vào `source`

Mỗi GPU chạy một tiến trình và xử lý một nửa số mẫu. Sau đó tiến trình chính ghép hai shard theo đúng thứ tự của manifest. Mô hình chỉ đọc hai trường `id` và `source`; trường `reference` không được đưa vào mô hình.

## 1. Chuẩn bị Kaggle

Trong **Notebook options** của Kaggle:

1. Chọn Accelerator: **GPU T4 x2**.
2. Bật Internet để tải mô hình ở lần chạy đầu.
3. Chạy các cell theo thứ tự từ trên xuống.

In [1]:
%pip install -q 'transformers>=4.46,<5.0' 'sentencepiece>=0.2' 'huggingface_hub>=0.24' 'tqdm>=4.66'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 2. Cấu hình

Bạn **chỉ bắt buộc sửa `INPUT_JSONL`**. Nếu gặp CUDA out of memory, giảm `BATCH_SIZE_PER_GPU` từ `4` xuống `2` hoặc `1` rồi chạy lại từ cell suy luận.

In [2]:
from pathlib import Path

# ==================== CHỈNH ĐƯỜNG DẪN NÀY ====================
INPUT_JSONL = '/kaggle/input/datasets/tranducthinh2006/file-test-core-2000/test_core_2000.jsonl'
# =============================================================

OUTPUT_JSONL = '/kaggle/working/vit5_base_beam4_lp1.1_max128_nr3_test_core_2000.jsonl'
MODEL_ID = 'VietAI/vit5-base-vietnews-summarization'
SYSTEM_NAME = 'vit5_base'
CONFIG_ID = 'beam4_lp1.1_max128_nr3'
SEED = 42
EXPECTED_NUM_ROWS = 2000
MAX_SOURCE_LENGTH = 512
BATCH_SIZE_PER_GPU = 4
HF_CACHE_DIR = '/kaggle/working/huggingface_cache'
INFERENCE_SCRIPT = Path('/kaggle/working/run_vit5_base_2gpu.py')

## 3. Kiểm tra GPU và manifest đầu vào

Cell này dừng sớm nếu Kaggle chưa bật đúng hai GPU, file không có đúng 2.000 dòng, ID bị trùng hoặc thiếu `source`.

In [3]:
import json
import torch
import transformers

input_path = Path(INPUT_JSONL)
assert input_path.is_file(), f'Không tìm thấy INPUT_JSONL: {input_path}'
assert torch.cuda.is_available(), 'Kaggle chưa bật GPU.'
assert torch.cuda.device_count() == 2, (
    f'Cần đúng 2 GPU nhưng hiện chỉ thấy {torch.cuda.device_count()}. '
    'Hãy chọn Accelerator = GPU T4 x2 trong Notebook options.'
)

rows = []
seen_ids = set()
with input_path.open('r', encoding='utf-8') as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            raise ValueError(f'Dòng {line_no} rỗng; JSONL không được có dòng rỗng.')
        try:
            row = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f'JSON không hợp lệ ở dòng {line_no}: {exc}') from exc
        sample_id = row.get('id')
        source = row.get('source')
        if not isinstance(sample_id, str) or not sample_id.strip():
            raise ValueError(f'id không hợp lệ ở dòng {line_no}.')
        if sample_id in seen_ids:
            raise ValueError(f'ID bị trùng: {sample_id}')
        if not isinstance(source, str) or not source.strip():
            raise ValueError(f'source rỗng hoặc không phải chuỗi tại id={sample_id}.')
        seen_ids.add(sample_id)
        rows.append(row)

assert len(rows) == EXPECTED_NUM_ROWS, (
    f'Cần {EXPECTED_NUM_ROWS} mẫu nhưng file có {len(rows)} mẫu.'
)

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
for gpu_index in range(torch.cuda.device_count()):
    print(f'GPU {gpu_index}:', torch.cuda.get_device_name(gpu_index))
print('Số mẫu:', len(rows))
print('ID đầu/cuối:', rows[0]['id'], '/', rows[-1]['id'])
print('Manifest hợp lệ.')

PyTorch: 2.10.0+cu128
Transformers: 4.57.6
GPU 0: Tesla T4
GPU 1: Tesla T4
Số mẫu: 2000
ID đầu/cuối: test_000007 / test_024693
Manifest hợp lệ.


## 4. Tải đúng các file cần thiết của mô hình

Mô hình được tải một lần trước khi mở hai tiến trình GPU, tránh việc hai tiến trình cùng tải một checkpoint.

In [4]:
from huggingface_hub import snapshot_download

LOCAL_MODEL_DIR = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    allow_patterns=[
        'config.json',
        'generation_config.json',
        'pytorch_model.bin',
        'spiece.model',
        'tokenizer.json',
        'tokenizer_config.json',
        'special_tokens_map.json',
    ],
)
print('Đã chuẩn bị mô hình tại:', LOCAL_MODEL_DIR)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Đã chuẩn bị mô hình tại: /kaggle/working/huggingface_cache/models--VietAI--vit5-base-vietnews-summarization/snapshots/a54febd011c0a39ce49ceacd9225e63ef3a73ad3


## 5. Tạo chương trình suy luận hai GPU

Notebook tự tạo file Python phụ trong `/kaggle/working`. Mỗi rank nhận các vị trí xen kẽ (`0,2,4,...` và `1,3,5,...`), ghi shard riêng, rồi rank 0 ghép và sắp xếp theo vị trí ban đầu. Nếu một batch lỗi, chương trình tự thử lại từng mẫu trong batch để không làm mất toàn bộ tiến trình.

In [5]:
INFERENCE_SCRIPT.write_text(r'''
import argparse
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--model-path', required=True)
    parser.add_argument('--system-name', required=True)
    parser.add_argument('--config-id', required=True)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--expected-num-rows', type=int, default=2000)
    parser.add_argument('--batch-size', type=int, default=4)
    parser.add_argument('--max-source-length', type=int, default=512)
    parser.add_argument('--fp16', action='store_true')
    return parser.parse_args()


def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_manifest(path, expected_num_rows):
    rows = []
    seen_ids = set()
    with Path(path).open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                raise ValueError(f'Blank line at line {line_no}')
            row = json.loads(line)
            sample_id = row.get('id')
            source = row.get('source')
            if not isinstance(sample_id, str) or not sample_id.strip():
                raise ValueError(f'Invalid id at line {line_no}')
            if sample_id in seen_ids:
                raise ValueError(f'Duplicate id: {sample_id}')
            if not isinstance(source, str) or not source.strip():
                raise ValueError(f'Invalid source for id={sample_id}')
            seen_ids.add(sample_id)
            rows.append(row)
    if len(rows) != expected_num_rows:
        raise ValueError(
            f'Expected {expected_num_rows} rows, found {len(rows)}'
        )
    return rows


def safe_error(exc):
    message = ' '.join(str(exc).split())
    return f'{type(exc).__name__}: {message}'[:1000]


def make_record(index, row, system_name, config_id, prediction, error=None):
    prediction = prediction.strip() if isinstance(prediction, str) else ''
    if error is None and prediction:
        status = 'ok'
        error_value = None
    else:
        status = 'error'
        error_value = error or 'EMPTY_PREDICTION'
        prediction = ''
    return {
        '_index': index,
        'id': row['id'],
        'system': system_name,
        'config_id': config_id,
        'prediction': prediction,
        'status': status,
        'error': error_value,
    }


def generate_texts(model, tokenizer, texts, device, max_source_length):
    # Không thêm prefix: tokenizer nhận chính xác nội dung trường source.
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_source_length,
        return_tensors='pt',
    )
    encoded = {key: value.to(device, non_blocking=True) for key, value in encoded.items()}
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            num_beams=4,
            length_penalty=1.1,
            max_new_tokens=128,
            no_repeat_ngram_size=3,
            do_sample=False,
            early_stopping=True,
        )
    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )


def write_jsonl_atomic(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = Path(str(path) + '.tmp')
    with temp_path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    os.replace(temp_path, path)


def merge_shards(output_path, world_size, rows):
    all_records = []
    for rank in range(world_size):
        shard_path = Path(f'{output_path}.rank{rank}.jsonl')
        if not shard_path.is_file():
            raise FileNotFoundError(f'Missing shard: {shard_path}')
        with shard_path.open('r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    all_records.append(json.loads(line))

    if len(all_records) != len(rows):
        raise ValueError(
            f'Expected {len(rows)} predictions, found {len(all_records)}'
        )
    all_records.sort(key=lambda record: record['_index'])
    actual_indices = [record['_index'] for record in all_records]
    expected_indices = list(range(len(rows)))
    if actual_indices != expected_indices:
        raise ValueError('Missing or duplicate internal indices after merge')
    if [record['id'] for record in all_records] != [row['id'] for row in rows]:
        raise ValueError('Prediction IDs do not match manifest order')

    final_records = []
    for record in all_records:
        final_records.append({
            'id': record['id'],
            'system': record['system'],
            'config_id': record['config_id'],
            'prediction': record['prediction'],
            'status': record['status'],
            'error': record['error'],
        })
    write_jsonl_atomic(output_path, final_records)


def main():
    args = parse_args()
    rank = int(os.environ['RANK'])
    local_rank = int(os.environ['LOCAL_RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    if world_size != 2:
        raise RuntimeError(f'This script requires exactly 2 processes, got {world_size}')

    torch.cuda.set_device(local_rank)
    device = torch.device(f'cuda:{local_rank}')
    dist.init_process_group(backend='nccl')
    seed_everything(args.seed)

    rows = load_manifest(args.input, args.expected_num_rows)
    local_items = [(index, rows[index]) for index in range(rank, len(rows), world_size)]

    tokenizer = AutoTokenizer.from_pretrained(args.model_path)
    dtype = torch.float16 if args.fp16 else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(
        args.model_path,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    ).to(device)
    model.eval()

    local_records = []
    batch_starts = range(0, len(local_items), args.batch_size)
    progress = tqdm(
        batch_starts,
        total=(len(local_items) + args.batch_size - 1) // args.batch_size,
        desc=f'GPU {rank}',
        position=rank,
    )
    for start in progress:
        batch = local_items[start:start + args.batch_size]
        texts = [row['source'] for _, row in batch]
        try:
            predictions = generate_texts(
                model, tokenizer, texts, device, args.max_source_length
            )
            if len(predictions) != len(batch):
                raise RuntimeError('Generated batch has an unexpected size')
            for (index, row), prediction in zip(batch, predictions):
                local_records.append(make_record(
                    index, row, args.system_name, args.config_id, prediction
                ))
        except Exception as batch_exc:
            if isinstance(batch_exc, torch.cuda.OutOfMemoryError):
                torch.cuda.empty_cache()
            # Thử lại từng mẫu để cô lập lỗi của batch.
            for index, row in batch:
                try:
                    prediction = generate_texts(
                        model, tokenizer, [row['source']], device, args.max_source_length
                    )[0]
                    local_records.append(make_record(
                        index, row, args.system_name, args.config_id, prediction
                    ))
                except Exception as sample_exc:
                    if isinstance(sample_exc, torch.cuda.OutOfMemoryError):
                        torch.cuda.empty_cache()
                    local_records.append(make_record(
                        index, row, args.system_name, args.config_id, '', safe_error(sample_exc)
                    ))

    shard_path = Path(f'{args.output}.rank{rank}.jsonl')
    write_jsonl_atomic(shard_path, local_records)
    dist.barrier()

    if rank == 0:
        merge_shards(args.output, world_size, rows)
        print(f'Final JSONL: {args.output}')

    dist.barrier()
    if rank == 0:
        for shard_rank in range(world_size):
            Path(f'{args.output}.rank{shard_rank}.jsonl').unlink(missing_ok=True)
    dist.destroy_process_group()


if __name__ == '__main__':
    main()
''', encoding='utf-8')

print('Đã tạo:', INFERENCE_SCRIPT)

Đã tạo: /kaggle/working/run_vit5_base_2gpu.py


## 6. Chạy suy luận trên 2 GPU T4

Mỗi GPU tải một bản ViT5-Base ở FP16 và xử lý khoảng 1.000 mẫu. Beam search là xác định (`do_sample=False`); seed 42 vẫn được đặt trên cả hai tiến trình để khóa toàn bộ nguồn ngẫu nhiên có thể phát sinh.

In [6]:
import os
import subprocess
import sys

output_path = Path(OUTPUT_JSONL)
assert output_path.resolve() != input_path.resolve(), 'Output không được trùng với input.'

command = [
    sys.executable, '-m', 'torch.distributed.run',
    '--standalone',
    '--nproc_per_node=2',
    str(INFERENCE_SCRIPT),
    '--input', str(input_path),
    '--output', str(output_path),
    '--model-path', str(LOCAL_MODEL_DIR),
    '--system-name', SYSTEM_NAME,
    '--config-id', CONFIG_ID,
    '--seed', str(SEED),
    '--expected-num-rows', str(EXPECTED_NUM_ROWS),
    '--batch-size', str(BATCH_SIZE_PER_GPU),
    '--max-source-length', str(MAX_SOURCE_LENGTH),
    '--fp16',
]

run_env = os.environ.copy()
run_env['TOKENIZERS_PARALLELISM'] = 'false'
run_env['OMP_NUM_THREADS'] = '1'
run_env['PYTHONUNBUFFERED'] = '1'

print('Bắt đầu suy luận bằng 2 GPU...')
subprocess.run(command, env=run_env, check=True)
print('Đã hoàn thành suy luận.')

Bắt đầu suy luận bằng 2 GPU...


[W811 16:15:44.548279156 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W811 16:15:58.710220145 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W811 16:15:58.710826315 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
GPU 0: 100%|██████████| 250/250 [07:29<00:00,  1.80s/it]
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W811 16:23:58.689559782 ProcessGroupNCCL.cpp:5138] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


Final JSONL: /kaggle/working/vit5_base_beam4_lp1.1_max128_nr3_test_core_2000.jsonl
Đã hoàn thành suy luận.


## 7. Kiểm tra bắt buộc và tải file output

Cell cuối xác nhận schema chính xác, đủ 2.000 ID, đúng thứ tự, không có ID trùng và không có mẫu lỗi. Nếu có lỗi, cell sẽ hiển thị một số lỗi rồi dừng để bạn sửa và chạy lại.

In [7]:
import hashlib
from IPython.display import FileLink, display

required_keys = {'id', 'system', 'config_id', 'prediction', 'status', 'error'}
predictions = []
with output_path.open('r', encoding='utf-8') as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            raise ValueError(f'Output có dòng rỗng tại dòng {line_no}.')
        record = json.loads(line)
        if set(record) != required_keys:
            raise ValueError(
                f'Schema sai tại dòng {line_no}: {sorted(record)}'
            )
        predictions.append(record)

input_ids = [row['id'] for row in rows]
output_ids = [record['id'] for record in predictions]
assert len(predictions) == EXPECTED_NUM_ROWS
assert len(output_ids) == len(set(output_ids)), 'Output có ID trùng.'
assert output_ids == input_ids, 'ID hoặc thứ tự output không khớp manifest.'
assert all(record['system'] == SYSTEM_NAME for record in predictions)
assert all(record['config_id'] == CONFIG_ID for record in predictions)
assert all(record['status'] in {'ok', 'error'} for record in predictions)
assert all(
    record['prediction'].strip() and record['error'] is None
    for record in predictions if record['status'] == 'ok'
)

errors = [record for record in predictions if record['status'] == 'error']
if errors:
    print('Một số lỗi đầu tiên:')
    for record in errors[:10]:
        print(record['id'], '->', record['error'])
assert not errors, (
    f'Có {len(errors)} mẫu lỗi. Hãy giảm BATCH_SIZE_PER_GPU nếu là OOM rồi chạy lại.'
)

sha256 = hashlib.sha256(output_path.read_bytes()).hexdigest()
print('Số prediction:', len(predictions))
print('Số mẫu thành công:', len(predictions) - len(errors))
print('Số mẫu lỗi:', len(errors))
print('SHA-256:', sha256)
print('Ví dụ 3 prediction đầu:')
for record in predictions[:3]:
    print(json.dumps(record, ensure_ascii=False)[:500])

display(FileLink(str(output_path)))

Số prediction: 2000
Số mẫu thành công: 2000
Số mẫu lỗi: 0
SHA-256: 913c8056ff09c0328d9dee76260eb674a10f64d582453a5a0f69b120bfaf71ae
Ví dụ 3 prediction đầu:
{"id": "test_000007", "system": "vit5_base", "config_id": "beam4_lp1.1_max128_nr3", "prediction": "Đó là nhận định của Thứ trưởng Bộ Giáo dục và Đào tạo Hoàng Minh Sơn khi trao đổi với báo chí bên hành lang Quốc hội sáng 12/6.", "status": "ok", "error": null}
{"id": "test_000014", "system": "vit5_base", "config_id": "beam4_lp1.1_max128_nr3", "prediction": "Tối 27/3, Ngô Thanh Vân tổ chức đám cưới với bạn trai Việt kiều tại Hà Nội. Cả hai đã có buổi gặp gỡ, trò chuyện với báo Người Đưa Tin để chuẩn bị cho ngày trọng đại của cô và Huy.", "status": "ok", "error": null}
{"id": "test_000027", "system": "vit5_base", "config_id": "beam4_lp1.1_max128_nr3", "prediction": "16 tấn vật tư y tế sẽ được chuyển tới Burkina Faso của Tây Phi để giúp đỡ các quốc gia bị nhiễm HIV.", "status": "ok", "error": null}


/kaggle/working/vit5_base_beam4_lp1.1_max128_nr3_test_core_2000.jsonl